<a href="https://colab.research.google.com/github/BassemRamdan/AI-Resume-Intelligence/blob/main/notebooks/03_OCR.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Phase 3: Layout & OCR Extraction
**Goal:** Prepare unstructured PDFs into Structured Layout Datasets (Text + Bounding Boxes + Page ID) required for Computer Vision Layout Modeling (e.g., LayoutLMv3).

**Strategy:**
- Since 2483/2484 PDFs are digital, we use `PyMuPDF` (`fitz`) to extract words and bounding boxes directly from PDF geometry without expensive rendering.
- For the 1 scanned PDF (or any page yielding 0 words), we fallback to rendering that page and running `Tesseract OCR`.
- We store the layout dynamically in JSON/HuggingFace Datasets.

In [ ]:
# Install Tesseract and system dependencies for image processing
!apt-get update
!apt-get install -y tesseract-ocr poppler-utils

# Install Python packages
!pip install pymupdf pytesseract pdf2image huggingface_hub pandas tqdm datasets

In [ ]:
import os
import fitz  # PyMuPDF
import pytesseract
from pdf2image import convert_from_path
import pandas as pd
import json
from huggingface_hub import snapshot_download
from tqdm.auto import tqdm

# Download the raw dataset again (or assume it's mounted)
print("Downloading dataset files from Hugging Face...")
dataset_path = snapshot_download(repo_id="BassemRamdan/data", repo_type="dataset")

categories = [d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d)) and not d.startswith('.git')]

In [ ]:
def extract_layout_pymupdf(pdf_path):
    """Extracts layout using PyMuPDF. Returns a list of pages with words and bboxes."""
    doc = fitz.open(pdf_path)
    pages_data = []
    
    for page_num in range(len(doc)):
        page = doc[page_num]
        # Get the page dimensions
        rect = page.rect
        page_width, page_height = rect.width, rect.height
        
        # get_text("words") returns tuples: (x0, y0, x1, y1, "word", block_no, line_no, word_no)
        words = page.get_text("words")
        
        page_words = []
        for w in words:
            x0, y0, x1, y1, text = w[0], w[1], w[2], w[3], w[4]
            page_words.append({
                "text": text,
                "bbox": [float(x0), float(y0), float(x1), float(y1)]
            })
            
        pages_data.append({
            "page_num": page_num + 1,
            "width": float(page_width),
            "height": float(page_height),
            "words": page_words
        })
        
    doc.close()
    return pages_data

def extract_layout_tesseract(pdf_path, page_num_to_ocr=None):
    """Fallback: Renders PDF to image and extracts bounding boxes using Tesseract."""
    # Convert only the specific page (1-indexed)
    if page_num_to_ocr:
        images = convert_from_path(pdf_path, first_page=page_num_to_ocr, last_page=page_num_to_ocr)
    else:
        images = convert_from_path(pdf_path)
        
    pages_data = []
    
    for i, img in enumerate(images):
        width, height = img.size
        # Get OCR data (includes bounding boxes)
        ocr_df = pytesseract.image_to_data(img, output_type=pytesseract.Output.DATAFRAME)
        
        # Filter out empty text (confidence < 0 or empty strings)
        ocr_df = ocr_df.dropna(subset=['text'])
        ocr_df = ocr_df[ocr_df['text'].str.strip() != '']
        
        page_words = []
        for _, row in ocr_df.iterrows():
            x0 = row['left']
            y0 = row['top']
            x1 = x0 + row['width']
            y1 = y0 + row['height']
            page_words.append({
                "text": str(row['text']),
                "bbox": [float(x0), float(y0), float(x1), float(y1)]
            })
            
        pages_data.append({
            "page_num": page_num_to_ocr if page_num_to_ocr else i + 1,
            "width": float(width),
            "height": float(height),
            "words": page_words
        })
        
    return pages_data

In [ ]:
print("Starting Full Layout Extraction Pipeline...")
structured_resumes = []
ocr_used_count = 0

for category in tqdm(categories, desc="Categories"):
    cat_path = os.path.join(dataset_path, category)
    for filename in os.listdir(cat_path):
        if filename.lower().endswith('.pdf'):
            file_path = os.path.join(cat_path, filename)
            
            try:
                # 1. Try PyMuPDF Digital Extraction First
                pages_data = extract_layout_pymupdf(file_path)
                
                # 2. Check if any page is just an image (0 words extracted)
                for page in pages_data:
                    if len(page['words']) == 0:
                        # Fallback to Tesseract OCR for this specific page
                        print(f"\nNo text found natively on {category}/{filename} Page {page['page_num']}. Using Tesseract OCR...")
                        ocr_pages = extract_layout_tesseract(file_path, page_num_to_ocr=page['page_num'])
                        if ocr_pages:
                            page['words'] = ocr_pages[0]['words']
                            page['width'] = ocr_pages[0]['width']
                            page['height'] = ocr_pages[0]['height']
                        ocr_used_count += 1
                
                structured_resumes.append({
                    "filename": filename,
                    "category": category,
                    "pages": pages_data
                })
                
            except Exception as e:
                print(f"\nError processing {file_path}: {e}")

print(f"\nExtraction Complete. Tesseract Fallback was used {ocr_used_count} times.")

In [ ]:
# Export structured layout data to a JSON file (This is our Dataset for Phase 4)
output_file = "resume_layout_dataset.json"
with open(output_file, 'w', encoding='utf-8') as f:
    json.dump(structured_resumes, f, ensure_ascii=False)

print(f"Saved layout data for {len(structured_resumes)} resumes to {output_file}.")
print("Size of JSON file on disk (MB):", os.path.getsize(output_file) / (1024 * 1024))

# Optional: Upload this JSON to Hugging Face Hub as a new layout dataset
# from huggingface_hub import HfApi
# api = HfApi()
# api.upload_file(
#     path_or_fileobj=output_file,
#     path_in_repo="resume_layout_dataset.json",
#     repo_id="BassemRamdan/resume-layout-data",
#     repo_type="dataset"
# )